# Pruebas de claves de publicaciones Silver

Pruebas en memoria. No escriben tablas ni modifican los notebooks ejecutados.


In [1]:
import json
import sys
from pathlib import Path
from pyspark.sql import functions as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config' / 'silver.json').is_file()), None)
if root is None:
    raise FileNotFoundError('No se encuentra config/silver.json')
sys.path.insert(0, str(root / 'scripts'))
from silver_quality import transform_publicaciones, transform_mediciones
config = json.loads((root / 'config' / 'silver.json').read_text(encoding='utf-8'))
bronze = json.loads((root / config['bronze_config']).read_text(encoding='utf-8'))
pub = spark.table(bronze['tables']['publicaciones'])
med = spark.table(bronze['tables']['mediciones'])


In [2]:
base = pub.filter((F.col('plataforma') == 'tiktok') & (F.col('video_id') == 'V000')).first().asDict()
copy = base.copy()
copy['_source_file'] = base['_source_file'] + '.identical-copy'
identical = spark.createDataFrame([base, copy], schema=pub.schema)
ok, rejected, conflict_keys = transform_publicaciones(identical, config)
assert ok.count() == 1 and rejected.count() == 1 and conflict_keys.count() == 0
assert rejected.first()['reason_codes'] == ['IDENTICAL_PUBLICATION_COPY']

changed = base.copy()
changed['fecha_publicacion'] = '2026-10-31'
changed['_source_file'] = base['_source_file'] + '.conflict'
conflicting = spark.createDataFrame([base, changed], schema=pub.schema)
ok2, rejected2, conflict_keys2 = transform_publicaciones(conflicting, config)
assert ok2.count() == 0 and rejected2.count() == 2 and conflict_keys2.count() == 1
assert all('CONFLICTING_PUBLICATION' in r['reason_codes'] for r in rejected2.collect())

measurement = med.filter((F.col('plataforma') == 'tiktok') & (F.col('video_id') == 'V000')).limit(1)
med_ok, med_rejected = transform_mediciones(measurement, ok2, conflict_keys2, config)
assert med_ok.count() == 0 and med_rejected.count() == 1
assert 'AMBIGUOUS_PUBLICATION' in med_rejected.first()['reason_codes']
print('Copia idéntica apartada; publicación conflictiva y medición ambigua rechazadas.')


Copia idéntica apartada; publicación conflictiva y medición ambigua rechazadas.
